## QWEN for Sentimnet / Emotion Analysis

In [1]:
import pandas as pd
import requests
from tqdm import tqdm

bsky = pd.read_csv('clean/bsky_posts_clean.csv')
sample = bsky.sample(5000, random_state=42)
print(f"Sample: {len(sample):,} posts")

Sample: 5,000 posts


In [2]:
def qwen_classify(text):
    try:
        r = requests.post('http://localhost:11434/api/generate', json={
            'model': 'qwen2.5:14b',
            'prompt': f'Classify the sentiment of this social media post about AI. Respond with ONLY one word: positive, negative, or neutral.\n\nPost: {str(text)[:500]}\n\nSentiment:',
            'stream': False
        }, timeout=60)
        response = r.json()['response'].strip().lower()
        if 'positive' in response: return 'positive'
        if 'negative' in response: return 'negative'
        return 'neutral'
    except:
        return None

# test on one post first
test = sample['text'].iloc[0]
print(f"Text: {test[:100]}")
print(f"Qwen says: {qwen_classify(test)}")

Text: I think companies should stop doing AI slop for their graphics designs and advertisements. Just seen
Qwen says: negative


In [6]:
results = []
for text in tqdm(sample['text'].tolist(), desc='Qwen'):
    results.append(qwen_classify(text))

sample['qwen_sentiment'] = results
sample.to_csv('clean/qwen_sample_sentiment.csv', index=False)

print(f"\nDistribution:")
print(sample['qwen_sentiment'].value_counts())
print(f"\nNull/failed: {sample['qwen_sentiment'].isna().sum()}")

Qwen: 100%|███████████████████████████| 5000/5000 [12:51:37<00:00,  9.26s/it]



Distribution:
qwen_sentiment
negative    2330
neutral     2004
positive     658
Name: count, dtype: int64

Null/failed: 8


In [ ]:
## Compute Log (manual)

# Mac Qwen started: ~22:25 BST, 2 Aug 2026

In [7]:
import time
print(f"Qwen finished at: {time.strftime('%Y-%m-%d %H:%M:%S')}")

Qwen finished at: 2026-08-03 11:17:31


In [ ]:
sample = bsky.sample(1000, random_state=42)

## Emotion 

In [3]:
def qwen_emotion(text):
    try:
        r = requests.post('http://localhost:11434/api/generate', json={
            'model': 'qwen2.5:14b',
            'prompt': f'What emotions are expressed in this social media post about AI? Choose ALL that apply from this list ONLY: anger, anticipation, disgust, fear, joy, love, optimism, pessimism, sadness, surprise, trust, neutral.\n\nRespond with ONLY the emotion words separated by commas, nothing else.\n\nPost: {str(text)[:500]}\n\nEmotions:',
            'stream': False
        }, timeout=60)
        data = r.json()
        if 'error' in data:
            return None
        response = data['response'].strip().lower()
        valid = {'anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust', 'neutral'}
        emotions = [e.strip() for e in response.split(',') if e.strip() in valid]
        return '|'.join(emotions) if emotions else 'neutral'
    except Exception as e:
        print(f"Error: {e}")
        return None

# test one post
test = sample['text'].iloc[0]
print(f"Text: {test[:100]}")
print(f"Emotions: {qwen_emotion(test)}")

Text: I think companies should stop doing AI slop for their graphics designs and advertisements. Just seen
Emotions: disgust|sadness|fear


In [4]:
emotion_results = []
for text in tqdm(sample['text'].tolist(), desc='Qwen Emotion'):
    emotion_results.append(qwen_emotion(text))

sample['qwen_emotions'] = emotion_results
sample.to_csv('clean/qwen_sample_full.csv', index=False)

print(f"\nNull/failed: {sample['qwen_emotions'].isna().sum()}")
from collections import Counter
counts = Counter(e for elist in sample['qwen_emotions'].dropna() for e in elist.split('|'))
print("Emotion distribution:")
for emotion, count in counts.most_common():
    print(f"  {emotion}: {count}")

Qwen Emotion:   7%|████████▌                                                                                                                 | 350/5000 [12:26<25:30:10, 19.74s/it]

Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)


Qwen Emotion:  22%|██████████████████████████▍                                                                                              | 1095/5000 [39:47<21:14:34, 19.58s/it]

Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)


Qwen Emotion:  34%|████████████████████████████████████████▎                                                                              | 1696/5000 [1:03:24<18:03:40, 19.68s/it]

Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)


Qwen Emotion: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [3:13:41<00:00,  2.32s/it]



Null/failed: 3
Emotion distribution:
  trust: 2387
  surprise: 1776
  optimism: 1464
  sadness: 1287
  fear: 1176
  neutral: 1041
  anger: 1014
  pessimism: 928
  joy: 892
  disgust: 801
  anticipation: 608
  love: 49


In [7]:
import os
for f in sorted(os.listdir('clean')):
    if 'qwen' in f.lower():
        print(f"  {f}: {os.path.getsize(f'clean/{f}')/1e6:.1f} MB")

  qwen_sample_full.csv: 3.1 MB
  qwen_sample_results.csv: 3.1 MB
  qwen_sample_sentiment.csv: 3.0 MB


In [8]:
sample = pd.read_csv('clean/qwen_sample_sentiment.csv')
print(sample.columns.tolist())
print(f"Rows: {len(sample):,}")

['post_uri', 'post_cid', 'text', 'created_at', 'reply_count', 'repost_count', 'like_count', 'quote_count', 'search_keyword', 'language', 'has_images', 'image_count', 'has_video', 'author_did', 'author_handle', 'author_display_name', 'author_avatar', 'author_description', 'platform', 'qwen_sentiment']
Rows: 5,000


In [9]:
print(sample['qwen_sentiment'].value_counts())
print(f"Nulls: {sample['qwen_sentiment'].isna().sum()}")


qwen_sentiment
negative    2330
neutral     2004
positive     658
Name: count, dtype: int64
Nulls: 8


In [10]:
import os
if os.path.exists('clean/qwen_sample_full.csv'):
    print("Emotion results exist")
else:
    print("Need to re-run emotion")

Emotion results exist


In [11]:
sample = pd.read_csv('clean/qwen_sample_full.csv')
print(f"Rows: {len(sample):,}")
print(f"\nSentiment:\n{sample['qwen_sentiment'].value_counts()}")
print(f"\nEmotion nulls: {sample['qwen_emotions'].isna().sum()}")

from collections import Counter
counts = Counter(e for labels in sample['qwen_emotions'].dropna() for e in labels.split('|'))
print(f"\nEmotions:")
for emotion, count in counts.most_common():
    print(f"  {emotion}: {count}")

Rows: 5,000


KeyError: 'qwen_sentiment'

In [12]:
sent = pd.read_csv('clean/qwen_sample_sentiment.csv')
full = pd.read_csv('clean/qwen_sample_full.csv')
print(f"Sentiment file columns: {sent.columns.tolist()}")
print(f"Full file columns: {full.columns.tolist()}")

Sentiment file columns: ['post_uri', 'post_cid', 'text', 'created_at', 'reply_count', 'repost_count', 'like_count', 'quote_count', 'search_keyword', 'language', 'has_images', 'image_count', 'has_video', 'author_did', 'author_handle', 'author_display_name', 'author_avatar', 'author_description', 'platform', 'qwen_sentiment']
Full file columns: ['post_uri', 'post_cid', 'text', 'created_at', 'reply_count', 'repost_count', 'like_count', 'quote_count', 'search_keyword', 'language', 'has_images', 'image_count', 'has_video', 'author_did', 'author_handle', 'author_display_name', 'author_avatar', 'author_description', 'platform', 'qwen_emotions']


In [13]:
sent = pd.read_csv('clean/qwen_sample_sentiment.csv')
full = pd.read_csv('clean/qwen_sample_full.csv')

# merge emotion into sentiment file
merged = sent.merge(full[['post_uri', 'qwen_emotions']], on='post_uri', how='left')
merged.to_csv('clean/qwen_sample_complete.csv', index=False)

print(f"Rows: {len(merged):,}")
print(f"\nSentiment:\n{merged['qwen_sentiment'].value_counts()}")
print(f"\nSentiment nulls: {merged['qwen_sentiment'].isna().sum()}")
print(f"Emotion nulls: {merged['qwen_emotions'].isna().sum()}")

from collections import Counter
counts = Counter(e for labels in merged['qwen_emotions'].dropna() for e in labels.split('|'))
print(f"\nEmotions:")
for emotion, count in counts.most_common():
    print(f"  {emotion}: {count}")

Rows: 5,000

Sentiment:
qwen_sentiment
negative    2330
neutral     2004
positive     658
Name: count, dtype: int64

Sentiment nulls: 8
Emotion nulls: 3

Emotions:
  trust: 2387
  surprise: 1776
  optimism: 1464
  sadness: 1287
  fear: 1176
  neutral: 1041
  anger: 1014
  pessimism: 928
  joy: 892
  disgust: 801
  anticipation: 608
  love: 49


In [6]:
# Save Qwen results — compare with RoBERTa later when both are on same machine
sample.to_csv('clean/qwen_sample_results.csv', index=False)
print(f"Saved {len(sample):,} posts with Qwen sentiment + emotion")
print(f"\nQwen sentiment: {sample['qwen_sentiment'].value_counts().to_dict()}")
print(f"Qwen nulls: {sample['qwen_sentiment'].isna().sum()}")

Saved 5,000 posts with Qwen sentiment + emotion


KeyError: 'qwen_sentiment'

In [5]:
# Load RoBERTa results for comparison
roberta_sent = pd.read_csv('clean/bsky_sentiment.csv')
merged = sample.merge(roberta_sent, on='post_uri', how='left')

agreement = (merged['qwen_sentiment'] == merged['label']).mean()
print(f"Sentiment agreement (RoBERTa vs Qwen): {agreement:.1%}")
print("\nCross-tabulation:")
print(pd.crosstab(merged['label'], merged['qwen_sentiment'], margins=True))

FileNotFoundError: [Errno 2] No such file or directory: 'clean/bsky_sentiment.csv'